# Hợp nhất LF → nhãn hữu dụng xác suất (label model)

Gộp phiếu mọi LF (`labels/votes/*.csv`) thành nhãn xác suất cho từng cặp (ảnh, tác vụ), rồi đối chiếu gold seed.
Mặc định: **biểu quyết có trọng số** trên phiếu đã dấu (1→+1, 0→−1, abstain→0); Snorkel `LabelModel` là tuỳ chọn khi mỗi tác vụ có ≥2 LF.
Khung hợp nhất & lựa chọn phương pháp: xem `docs/Labeling_Plan.md`. Schema phiếu & `fuse_votes`: xem `src/utils/lf_io.ipynb`.

## Cấu hình

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RUNNER = 'local'          # 'local' | 'kaggle'
THRESHOLD = 0.5           # prob_usable >= THRESHOLD -> nhan cung = 1
WEIGHT_MODE = 'uniform'   # 'uniform' | 'gold_alpha' (trong so = max(0, 2*alpha_hat-1) do tren gold seed)
USE_SNORKEL = False       # True: dung snorkel.LabelModel khi tac vu co >=2 LF (can cai snorkel)

print('runner:', RUNNER)
print('threshold:', THRESHOLD)
print('weight_mode:', WEIGHT_MODE)
print('use_snorkel:', USE_SNORKEL)

## Đường dẫn + module dùng chung (`src/utils/lf_io.ipynb`)

In [ ]:
def build_root(runner):
    if runner == 'local':
        candidates = [Path.cwd(), Path.cwd().parent]
        for cand in candidates:
            probe = cand / 'labels' / 'votes'
            if probe.exists():
                return cand
        raise SystemExit('Khong thay labels/votes — chay notebook trong repo coconut-iqa')
    if runner == 'kaggle':
        root = Path('/kaggle/input/coconut-iqa')
        if not (root / 'labels' / 'votes').exists():
            raise SystemExit('Khong thay /kaggle/input/coconut-iqa/labels/votes')
        return root
    raise SystemExit("RUNNER phai la 'local' hoac 'kaggle'")

ROOT = build_root(RUNNER)
VOTES_DIR = ROOT / 'labels' / 'votes'
GOLD_CSV = ROOT / 'gold_seed' / 'gold_seed_labels.csv'
FUSED_CSV = ROOT / 'labels' / 'fused' / 'usability_labels.csv'

UTILS = ROOT / 'src' / 'utils' / 'lf_io.ipynb'
if not UTILS.exists():
    raise SystemExit('Khong thay ' + str(UTILS))
get_ipython().run_line_magic('run', str(UTILS))

print('ROOT:', ROOT)
print('VOTES_DIR:', VOTES_DIR)
print('GOLD_CSV:', GOLD_CSV, '| ton tai:', GOLD_CSV.exists())
print('FUSED_CSV:', FUSED_CSV)

## 1. Nạp phiếu + trọng số mỗi LF theo tác vụ

In [ ]:
long_df, wide_df = fuse_votes(VOTES_DIR)
long_df = long_df.dropna(subset=['vote'])
long_df['vote'] = long_df['vote'].astype(int)

print('phieu (long):', len(long_df))
print('cap (lf, task):')
print(long_df.groupby(['lf', 'task']).size().reset_index(name='n').to_string(index=False))


def gold_alpha_weights(long_df, gold_csv):
    # Trong so = max(0, 2*alpha_hat-1); alpha_hat do tren phan gold seed cua tung (lf, task).
    weights = {}
    if not gold_csv.exists():
        return weights
    gold = pd.read_csv(gold_csv)
    for (lf, task), grp in long_df.groupby(['lf', 'task']):
        if task not in gold.columns:
            continue
        merged = grp.merge(gold[['image_id', task]], on='image_id', how='inner')
        if len(merged) == 0:
            continue
        y_true = merged[task].astype(int)
        y_vote = merged['vote'].astype(int)
        alpha = float((y_true == y_vote).mean())
        weights[(lf, task)] = max(0.0, 2.0 * alpha - 1.0)
    return weights


if WEIGHT_MODE == 'gold_alpha':
    WEIGHTS = gold_alpha_weights(long_df, GOLD_CSV)
    print('trong so gold_alpha:', WEIGHTS)
else:
    WEIGHTS = {}


def weight_of(lf, task):
    if WEIGHT_MODE == 'gold_alpha':
        return WEIGHTS.get((lf, task), 0.0)
    return 1.0

## 2. Hợp nhất → xác suất hữu dụng

Với mỗi (ảnh, tác vụ): $\text{score}=\sum_i w_i\,(2v_i-1)$ trên các LF không abstain; $\Pr(\text{hữu dụng})=\sigma(\text{score})$.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def fuse_group(grp):
    score = 0.0
    n_pos = 0
    n_neg = 0
    for r in grp.itertuples():
        w = weight_of(r.lf, r.task)
        signed = 2 * int(r.vote) - 1
        score = score + w * signed
        if int(r.vote) == 1:
            n_pos = n_pos + 1
        else:
            n_neg = n_neg + 1
    prob = sigmoid(score)
    return score, prob, n_pos, n_neg


rows = []
for (image_id, task), grp in long_df.groupby(['image_id', 'task']):
    score, prob, n_pos, n_neg = fuse_group(grp)
    row = {}
    row['image_id'] = image_id
    row['task'] = task
    row['n_votes'] = n_pos + n_neg
    row['n_pos'] = n_pos
    row['n_neg'] = n_neg
    row['score'] = round(score, 4)
    row['prob_usable'] = round(prob, 4)
    if prob >= THRESHOLD:
        row['label'] = 1
    else:
        row['label'] = 0
    rows.append(row)

fused = pd.DataFrame(rows)
print('cap (anh, tac vu) co nhan:', len(fused))
print(fused.groupby('task')['label'].agg(['count', 'mean']).rename(columns={'mean': 'ti_le_1'}).to_string())

## 3. Ghi nhãn hợp nhất `labels/fused/usability_labels.csv`

In [ ]:
FUSED_CSV.parent.mkdir(parents=True, exist_ok=True)
cols = ['image_id', 'task', 'n_votes', 'n_pos', 'n_neg', 'score', 'prob_usable', 'label']
fused[cols].to_csv(FUSED_CSV, index=False)
print('ghi:', FUSED_CSV, '|', len(fused), 'dong')

## 4. Đối chiếu gold seed (đánh giá nhãn hợp nhất)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import cohen_kappa_score

if not GOLD_CSV.exists():
    print('khong co gold seed -> bo qua danh gia')
else:
    gold = pd.read_csv(GOLD_CSV)
    eval_rows = []
    for task in sorted(fused['task'].unique()):
        if task not in gold.columns:
            continue
        sub = fused[fused['task'] == task]
        merged = sub.merge(gold[['image_id', task]], on='image_id', how='inner')
        if len(merged) == 0:
            continue
        y_true = merged[task].astype(int)
        y_pred = merged['label'].astype(int)
        rec = {}
        rec['task'] = task
        rec['n_overlap'] = len(merged)
        rec['ti_le_gold_1'] = round(float(y_true.mean()), 4)
        rec['accuracy'] = round(accuracy_score(y_true, y_pred), 4)
        rec['kappa'] = round(cohen_kappa_score(y_true, y_pred), 4)
        eval_rows.append(rec)
    if eval_rows:
        print(pd.DataFrame(eval_rows).to_string(index=False))
    else:
        print('khong co overlap voi gold seed')

## Giới hạn

Với tác vụ chỉ có **một LF** bỏ phiếu, hợp nhất suy biến về chính phiếu đó (prob ∈ {σ(−w), σ(w)}); hợp nhất chỉ có ý nghĩa khi nhiều LF cùng phủ một tác vụ.
`accuracy` trên gold seed bị lệch bởi base-rate (gold ~97% hữu dụng ở maturity/foliar) → đọc kèm $\kappa$. Xem mục Hạn chế `paper/Methodology.md` §3.9.
Snorkel `LabelModel` (ước lượng độ tin LF từ đồng thuận, không cần gold) là bản nâng cấp khi mỗi tác vụ có ≥2–3 LF.